<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/04_Foundations/statistics/04_regression_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regression Statistics

The statistical backbone of predictive modeling — understand what OLS actually optimizes, when assumptions break, and how regularization controls overfitting.

**Topics:** Simple/multiple linear regression, OLS derivation, LINE assumptions, residual diagnostics, logistic regression, Ridge/Lasso, R² and adjusted R²

## 1. Simple Linear Regression — OLS from Scratch

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)

# Dataset: advertising spend → sales
n = 100
ad_spend = np.random.uniform(10, 100, n)          # $10k–$100k spend
sales = 50 + 2.5 * ad_spend + np.random.normal(0, 15, n)  # true: β0=50, β1=2.5

# OLS closed-form solution: β = (XᵀX)⁻¹Xᵀy
def ols_fit(x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Fit simple linear regression via OLS."""
    x_mean, y_mean = x.mean(), y.mean()
    beta_1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
    beta_0 = y_mean - beta_1 * x_mean
    return beta_0, beta_1

b0, b1 = ols_fit(ad_spend, sales)
print('OLS from scratch (Simple Linear Regression):')
print(f'  Fitted: ŷ = {b0:.2f} + {b1:.3f}·x')
print(f'  True:   ŷ = 50 + 2.500·x')
print()

# Why OLS minimizes sum of squared residuals
y_pred = b0 + b1 * ad_spend
residuals = sales - y_pred
sse = np.sum(residuals ** 2)   # Sum of Squared Errors
sst = np.sum((sales - sales.mean()) ** 2)  # Total Sum of Squares
ssr = sst - sse                # Regression Sum of Squares

r2 = 1 - sse / sst
print(f'Variance decomposition:')
print(f'  SST (total) = {sst:.1f}')
print(f'  SSR (regression/explained) = {ssr:.1f}')
print(f'  SSE (error/residual) = {sse:.1f}')
print(f'  R² = SSR/SST = {r2:.4f}  ({r2*100:.1f}% of variance explained)')
print()

# Verify with scipy
result = stats.linregress(ad_spend, sales)
print(f'scipy.stats.linregress verification:')
print(f'  slope={result.slope:.3f}, intercept={result.intercept:.2f}, r²={result.rvalue**2:.4f}')
print(f'  p-value for slope: {result.pvalue:.6f} (is β1 ≠ 0?)')
print(f'  std error of slope: {result.stderr:.4f}')
print()
print('OLS properties (BLUE — Best Linear Unbiased Estimator under Gauss-Markov):')
for prop in ['Unbiased: E[β̂] = β', 'Efficient: minimum variance among all linear unbiased estimators',
             'Consistent: β̂ → β as n → ∞']:
    print(f'  ✓ {prop}')

## 2. Multiple Linear Regression — Matrix Form and Inference

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 200

# Generate features: house price prediction
sqft = np.random.normal(1800, 400, n)             # square feet
bedrooms = np.random.randint(1, 6, n).astype(float)
age = np.random.uniform(0, 50, n)                  # house age in years
price = (150 * sqft + 8000 * bedrooms - 500 * age
         + np.random.normal(0, 50_000, n))

# Matrix formulation: y = Xβ + ε, β̂ = (XᵀX)⁻¹Xᵀy
X = np.column_stack([np.ones(n), sqft, bedrooms, age])  # add intercept column
y = price

beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]  # numerically stable
y_pred = X @ beta_hat
residuals = y - y_pred

# Standard errors of coefficients
p = X.shape[1]          # number of parameters
mse = np.sum(residuals**2) / (n - p)  # mean squared error
cov_beta = mse * np.linalg.inv(X.T @ X)  # covariance of β̂
se_beta = np.sqrt(np.diag(cov_beta))

# t-statistics and p-values for each coefficient
t_stats = beta_hat / se_beta
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-p))

# 95% confidence intervals
t_crit = stats.t.ppf(0.975, df=n-p)
ci_lower = beta_hat - t_crit * se_beta
ci_upper = beta_hat + t_crit * se_beta

print('Multiple Linear Regression: Price = β0 + β1·sqft + β2·bedrooms + β3·age')
print(f'{"Feature":<12} {"Coef":>12} {"Std Err":>10} {"t":>8} {"p":>10} {"95% CI"}')
print('-' * 75)
names = ['Intercept', 'sqft', 'bedrooms', 'age']
true_coefs = [0, 150, 8000, -500]
for i, name in enumerate(names):
    sig = '***' if p_values[i] < 0.001 else '**' if p_values[i] < 0.01 else '*' if p_values[i] < 0.05 else ''
    print(f'{name:<12} {beta_hat[i]:>12.1f} {se_beta[i]:>10.1f} {t_stats[i]:>8.2f} {p_values[i]:>10.4f}{sig}  [{ci_lower[i]:.1f}, {ci_upper[i]:.1f}]')

print(f'\nTrue coefficients: β0≈0, β1=150, β2=8000, β3=-500')

# R² and adjusted R²
r2 = 1 - np.sum(residuals**2) / np.sum((y - y.mean())**2)
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p)
print(f'\nModel fit: R²={r2:.4f}, Adjusted R²={r2_adj:.4f}')
print('Adjusted R² penalizes for number of predictors — use for model comparison.')

## 3. LINE Assumptions and Residual Diagnostics

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)

# LINE assumptions
print('LINE Assumptions for Linear Regression:')
assumptions = [
    ('L — Linearity', 'E[y|x] is linear in parameters', 'Residual vs Fitted plot (should show no pattern)'),
    ('I — Independence', 'Errors are independent', 'Durbin-Watson test (for time series)'),
    ('N — Normality', 'Errors are normally distributed', 'QQ-plot or Shapiro-Wilk on residuals'),
    ('E — Equal variance', 'Homoscedasticity: Var(ε) = σ²', 'Scale-Location plot, Breusch-Pagan test'),
]
for assumption, meaning, diagnostic in assumptions:
    print(f'  {assumption}')
    print(f'    Meaning:    {meaning}')
    print(f'    Diagnostic: {diagnostic}')
print()

# Generate data with assumption violations to demonstrate diagnostics
n = 150
x = np.linspace(1, 10, n)

# Well-behaved residuals
y_good = 3 + 2 * x + np.random.normal(0, 1, n)

# Heteroscedastic: variance grows with x
y_hetero = 3 + 2 * x + np.random.normal(0, x * 0.5, n)

# Non-linear: quadratic true relationship but fitting linear
y_nonlin = 3 + 2 * x + 0.3 * x**2 + np.random.normal(0, 1, n)

def diagnose_residuals(x: np.ndarray, y: np.ndarray, name: str) -> None:
    """Run residual diagnostics for OLS regression."""
    # Fit OLS
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_pred = X @ beta
    residuals = y - y_pred

    # Normality test
    _, norm_p = stats.shapiro(residuals)

    # Homoscedasticity: correlate |residuals| with fitted values
    hetero_corr, hetero_p = stats.spearmanr(np.abs(residuals), y_pred)

    # Pattern in residuals: correlate residuals with fitted values
    pattern_corr, _ = stats.spearmanr(residuals**2, y_pred)

    print(f'{name}:')
    print(f'  Normality (Shapiro p): {norm_p:.4f} → {"OK" if norm_p > 0.05 else "VIOLATION"}')
    print(f'  Homoscedasticity (|res|~fitted, Spearman p): {hetero_p:.4f} → {"OK" if hetero_p > 0.05 else "VIOLATION"}')
    print(f'  Non-linearity (res²~fitted, Spearman r): {pattern_corr:.3f} → {"OK" if abs(pattern_corr) < 0.2 else "CHECK FOR PATTERN"}')
    print()

diagnose_residuals(x, y_good, 'Well-specified model')
diagnose_residuals(x, y_hetero, 'Heteroscedastic model')
diagnose_residuals(x, y_nonlin, 'Non-linear relationship (misspecified)')

print('Fixes for violations:')
fixes = [
    ('Non-linearity', 'Add polynomial terms, interaction terms, or use tree-based models'),
    ('Heteroscedasticity', 'Log-transform y, use WLS (weighted least squares), or robust SE'),
    ('Non-normality', 'CLT helps with large n; for small n use GLMs or transformations'),
    ('Multicollinearity', 'Check VIF > 10; remove correlated features or use Ridge regression'),
]
for issue, fix in fixes:
    print(f'  {issue:<20}: {fix}')

## 4. Logistic Regression — Binary Classification Statistics

In [ ]:
import numpy as np
from scipy import stats, optimize

np.random.seed(42)

# Generate binary outcome: P(churn | tenure, monthly_charges)
n = 500
tenure = np.random.exponential(scale=20, size=n).clip(0, 72)   # months
monthly = np.random.normal(65, 20, n).clip(20, 120)             # dollars

# True log-odds: negative tenure (loyal customers stay) + positive charges (expensive → churn)
log_odds_true = -0.05 * tenure + 0.04 * monthly - 1.5
p_churn = 1 / (1 + np.exp(-log_odds_true))
y = np.random.binomial(1, p_churn)

# Maximum Likelihood Estimation for logistic regression
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z.clip(-500, 500)))

def neg_log_likelihood(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    """Binary cross-entropy loss = negative log-likelihood."""
    p = sigmoid(X @ beta)
    p = p.clip(1e-10, 1 - 1e-10)
    return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

X = np.column_stack([np.ones(n), tenure, monthly])
result = optimize.minimize(neg_log_likelihood, x0=np.zeros(3), args=(X, y),
                            method='BFGS')
beta_mle = result.x

# Compute standard errors via Fisher information matrix
p_hat = sigmoid(X @ beta_mle)
W = np.diag(p_hat * (1 - p_hat))
fisher_info = X.T @ W @ X
se = np.sqrt(np.diag(np.linalg.inv(fisher_info)))
z_stats = beta_mle / se
p_vals = 2 * (1 - stats.norm.cdf(np.abs(z_stats)))

# Odds ratios: exp(β) = multiplicative change in odds per unit increase
odds_ratios = np.exp(beta_mle)

print('Logistic Regression: MLE Estimates')
print(f'{"Feature":<15} {"Coef":>8} {"True":>8} {"SE":>8} {"z":>7} {"p":>8} {"Odds Ratio":>12}')
print('-' * 70)
names = ['Intercept', 'tenure', 'monthly']
true_coefs = [-1.5, -0.05, 0.04]
for i, name in enumerate(names):
    sig = '***' if p_vals[i] < 0.001 else '**' if p_vals[i] < 0.01 else '*' if p_vals[i] < 0.05 else ''
    print(f'{name:<15} {beta_mle[i]:>8.3f} {true_coefs[i]:>8.3f} {se[i]:>8.3f} {z_stats[i]:>7.2f} {p_vals[i]:>8.4f}{sig} {odds_ratios[i]:>12.4f}')

print()
print('Interpreting odds ratios:')
print(f'  tenure:  OR={odds_ratios[1]:.4f} → each extra month reduces churn odds by {(1-odds_ratios[1])*100:.1f}%')
print(f'  monthly: OR={odds_ratios[2]:.4f} → each +$1 in charges increases churn odds by {(odds_ratios[2]-1)*100:.1f}%')
print()

# Model fit: null deviance vs residual deviance
p_hat = sigmoid(X @ beta_mle)
p_hat = p_hat.clip(1e-10, 1 - 1e-10)
null_ll = n * np.log(0.5)  # intercept-only model (uninformative)
full_ll = np.sum(y * np.log(p_hat) + (1 - y) * np.log(1 - p_hat))
mcfadden_r2 = 1 - full_ll / null_ll
print(f"McFadden's pseudo-R² = {mcfadden_r2:.4f}  (0.2–0.4 = excellent for logistic regression)")

## 5. Regularization — Ridge, Lasso, and the Bias-Variance Tradeoff

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)

# High-dimensional regression: many features, some irrelevant
n, p_true, p_total = 200, 5, 50
X_raw = np.random.randn(n, p_total)
true_beta = np.zeros(p_total)
true_beta[:p_true] = [3, -2, 1.5, -1, 0.8]  # only first 5 features matter
y = X_raw @ true_beta + np.random.normal(0, 1, n)

# Standardize features (critical for regularization)
X = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)

def ridge_regression(X: np.ndarray, y: np.ndarray, lam: float) -> np.ndarray:
    """Ridge: β̂_ridge = (XᵀX + λI)⁻¹Xᵀy."""
    n, p = X.shape
    return np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)

def soft_threshold(z: np.ndarray, threshold: float) -> np.ndarray:
    """Lasso coordinate descent update: soft-thresholding."""
    return np.sign(z) * np.maximum(np.abs(z) - threshold, 0)

def lasso_coordinate_descent(X: np.ndarray, y: np.ndarray, lam: float,
                               max_iter: int = 1000, tol: float = 1e-6) -> np.ndarray:
    """Lasso via coordinate descent."""
    n, p = X.shape
    beta = np.zeros(p)
    for _ in range(max_iter):
        beta_old = beta.copy()
        for j in range(p):
            r_j = y - X @ beta + X[:, j] * beta[j]  # partial residual
            z_j = X[:, j] @ r_j / n
            beta[j] = soft_threshold(np.array([z_j]), lam / n)[0]
        if np.max(np.abs(beta - beta_old)) < tol:
            break
    return beta

# Compare OLS vs Ridge vs Lasso
beta_ols = np.linalg.lstsq(X, y, rcond=None)[0]
beta_ridge = ridge_regression(X, y, lam=10)
beta_lasso = lasso_coordinate_descent(X, y, lam=0.5)

print(f'Coefficient comparison (true vs OLS vs Ridge(λ=10) vs Lasso(λ=0.5)):')
print(f'{"Feature":<10} {"True":>8} {"OLS":>8} {"Ridge":>8} {"Lasso":>8}')
print('-' * 45)
for i in range(p_total):
    is_true = i < p_true
    marker = '<-- signal' if is_true else ''
    print(f'x{i:<9} {true_beta[i]:>8.3f} {beta_ols[i]:>8.3f} {beta_ridge[i]:>8.3f} {beta_lasso[i]:>8.3f}  {marker}')

print()
print(f'Non-zero Lasso coefficients: {(beta_lasso != 0).sum()} / {p_total}  (Lasso does feature selection!)')
print(f'Non-zero Ridge coefficients: {(beta_ridge != 0).sum()} / {p_total}  (Ridge shrinks but never zeros)')
print()
print('Key differences:')
print('  Ridge (L2 penalty: λΣβⱼ²):  shrinks all coefficients, good for correlated features')
print('  Lasso (L1 penalty: λΣ|βⱼ|): forces sparse solutions via soft-thresholding')
print('  ElasticNet (α·L1 + (1-α)·L2): combines both — best of both worlds')
print()
print('Bias-variance tradeoff:')
print('  λ → 0: OLS (low bias, high variance — overfits)')
print('  λ → ∞: intercept-only (high bias, zero variance — underfits)')
print('  Optimal λ: cross-validate! Use CV to find the "Goldilocks" λ.')

## Exercises

1. **Multicollinearity:** Generate two features x1 and x2 with correlation ρ=0.95. Fit a regression with both. Compute the Variance Inflation Factor (VIF = 1/(1-R²_j) for each feature). Show how the standard errors explode with high VIF. Then apply Ridge regression and show how it stabilizes estimates.

2. **Cross-validation for λ:** Implement k-fold cross-validation (k=5) to select the optimal λ for Ridge regression. Plot the validation MSE vs log(λ) path. Mark the λ_min and λ_1se (largest λ within 1 SE of minimum). Explain why λ_1se is often preferred in practice.

3. **Logistic regression calibration:** Train a logistic regression on a credit default dataset. Plot a calibration curve (predicted probability bins vs actual default rate). Compute the Brier score. Apply Platt scaling to improve calibration and compare.